<a href="https://colab.research.google.com/github/jyizheng/my-study/blob/main/leetcode/all_O_1_ds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
class ItemNode:
    def __init__(self, id, score=1):
        self.id = id
        self.score = score
        self.prev = None
        self.next = None
        self.bucket = None   # back pointer to its bucket

class Bucket:
    def __init__(self, score):
        self.score = score
        self.items = {}     # id -> ItemNode
        self.prev = None
        self.next = None

    def add(self, node):
        self.items[node.id] = node
        node.bucket = self

    def remove(self, node):
        if node.id in self.items:
            del self.items[node.id]
        node.bucket = None

    def is_empty(self):
        return len(self.items) == 0

class PopularItemsTracker:
    def __init__(self):
        self.ids = {}       # id -> ItemNode
        self.scores = {}    # score -> Bucket
        # dummy head/tail for bucket list
        self.head = Bucket(float('-inf'))
        self.tail = Bucket(float('inf'))
        self.head.next = self.tail
        self.tail.prev = self.head

    def _insert_bucket_after(self, new_bucket, prev_bucket):
        nxt = prev_bucket.next
        prev_bucket.next = new_bucket
        new_bucket.prev = prev_bucket
        new_bucket.next = nxt
        nxt.prev = new_bucket

    def _remove_bucket(self, bucket):
        bucket.prev.next = bucket.next
        bucket.next.prev = bucket.prev
        del self.scores[bucket.score]

    def insert(self, id: str):
        if id in self.ids:
            node = self.ids[id]
            old_bucket = node.bucket
            old_bucket.remove(node)

            node.score += 1
            # move to next bucket
            if node.score not in self.scores:
                new_bucket = Bucket(node.score)
                self.scores[node.score] = new_bucket
                self._insert_bucket_after(new_bucket, old_bucket)
            new_bucket = self.scores[node.score]
            new_bucket.add(node)

            if old_bucket.is_empty():
                self._remove_bucket(old_bucket)
        else:
            node = ItemNode(id, 1)
            self.ids[id] = node
            if 1 not in self.scores:
                bucket = Bucket(1)
                self.scores[1] = bucket
                self._insert_bucket_after(bucket, self.head)
            self.scores[1].add(node)

    def top_k(self, k: int):
        res = []
        bucket = self.tail.prev
        while bucket != self.head and len(res) < k:
            for id, node in bucket.items.items():
                res.append((id, node.score))
                if len(res) == k:
                    break
            bucket = bucket.prev
        return res


# ----------------- 测试代码 -----------------
def test_tracker():
    tracker = PopularItemsTracker()

    # Case 1: 基本插入
    tracker.insert("a")
    tracker.insert("b")
    print("After inserting a, b:", tracker.top_k(5))  # [('a',1), ('b',1)]

    # Case 2: 重复插入
    tracker.insert("a")
    tracker.insert("a")
    tracker.insert("c")
    tracker.insert("a")
    print("After multiple inserts:", tracker.top_k(3))  # [('a',4), ('c',1), ('b',1)]

    # Case 3: 稀疏分数
    for _ in range(9):  # b -> score=10
        tracker.insert("b")
    print("Sparse scores test:", tracker.top_k(3))  # [('b',10), ('a',4), ...]

    for _ in range(90):  # c -> score=100
        tracker.insert("c")
    print("More sparse test:", tracker.top_k(3))  # [('c',100), ('b',10), ('a',4)]

    # Case 4: k > 总元素数
    print("Top 10 (larger than total):", tracker.top_k(10))

    # Case 5: 空 tracker
    empty = PopularItemsTracker()
    print("Empty tracker:", empty.top_k(5))


if __name__ == "__main__":
    test_tracker()



After inserting a, b: [('a', 1), ('b', 1)]
After multiple inserts: [('a', 4), ('b', 1), ('c', 1)]
Sparse scores test: [('b', 10), ('a', 4), ('c', 1)]
More sparse test: [('c', 91), ('b', 10), ('a', 4)]
Top 10 (larger than total): [('c', 91), ('b', 10), ('a', 4)]
Empty tracker: []
